# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shivanilokh/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

## My Rule

A page should be reviewed if it has high search visibility, has not been updated for a long time, and is showing signs of declining performance. Older pages with good impressions are given higher priority because refreshing them is more likely to improve traffic.

### Reason Codes

- STALE_CONTENT
- HIGH_IMPRESSIONS
- LOW_CTR
- DECLINING_PERFORMANCE
- REFRESH_RECOMMENDED

In [12]:
import pandas as pd

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

df.head()

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [13]:
print(df.columns.tolist())

['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


In [14]:
# Signal Check 1: Days since last update vs impressions

df.groupby("freshness_tier").agg(
    n=("content_id", "count"),
    avg_impressions=("impressions_90d", "mean"),
    avg_ctr=("ctr", "mean")
).reset_index()

,freshness_tier,n,avg_impressions,avg_ctr
0,0-30,20480,4199.614062,0.609021
1,181+,174,1172.448276,3.693276
2,31-90,175,6506.748571,0.117543
3,91-180,9171,7486.665140,0.238367


In [15]:
# Signal Check 2: Position tier vs CTR

df.groupby("position_tier").agg(
    n=("content_id", "count"),
    avg_ctr=("ctr", "mean"),
    avg_position=("avg_position", "mean")
).reset_index()

,position_tier,n,avg_ctr,avg_position
0,deep,1319,0.150212,63.664822
1,page_1,11814,0.652467,6.573269
2,page_3_5,7242,0.222484,30.673888
3,striking,7304,0.323239,14.259953
4,top_3,2321,1.483611,1.010728


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [16]:
import os

baseline = df.copy()

baseline["score"] = (
    (baseline["days_since_last_update"] >= 180).astype(int) * 3
    + (baseline["impressions_90d"] >= 5000).astype(int) * 2
    + (baseline["ctr"] < 0.5).astype(int)
)

def reason_code(row):
    reasons = []

    if row["days_since_last_update"] >= 180:
        reasons.append("STALE_CONTENT")

    if row["impressions_90d"] >= 5000:
        reasons.append("HIGH_IMPRESSIONS")

    if row["ctr"] < 0.5:
        reasons.append("LOW_CTR")

    if len(reasons) == 0:
        reasons.append("NO_ACTION")

    return ",".join(reasons)

baseline["reason_code"] = baseline.apply(reason_code, axis=1)

baseline["action"] = baseline["score"].apply(
    lambda x: "REFRESH_CONTENT" if x >= 5 else "MONITOR"
)

baseline = baseline.sort_values("score", ascending=False)

os.makedirs("../outputs", exist_ok=True)

baseline.to_csv("../outputs/baseline_action_score.csv", index=False)

baseline[["content_id", "score", "reason_code", "action"]].head(10)

,content_id,score,reason_code,action
21268,content_0a91db491d14,6,"STALE_CONTENT,HIGH_IMPRESSIONS,LOW_CTR",REFRESH_CONTENT
16751,content_cf56e2e2e282,6,"STALE_CONTENT,HIGH_IMPRESSIONS,LOW_CTR",REFRESH_CONTENT
16514,content_7368877ea310,6,"STALE_CONTENT,HIGH_IMPRESSIONS,LOW_CTR",REFRESH_CONTENT
12045,content_c2d929d83eaa,6,"STALE_CONTENT,HIGH_IMPRESSIONS,LOW_CTR",REFRESH_CONTENT
11489,content_5feee3994adb,6,"STALE_CONTENT,HIGH_IMPRESSIONS,LOW_CTR",REFRESH_CONTENT
7021,content_1bfaa38ff26c,6,"STALE_CONTENT,HIGH_IMPRESSIONS,LOW_CTR",REFRESH_CONTENT
20924,content_277eeb6d46cc,4,"STALE_CONTENT,LOW_CTR",MONITOR
21201,content_afd26a07382d,4,"STALE_CONTENT,LOW_CTR",MONITOR
22980,content_f328d0e4e22b,4,"STALE_CONTENT,LOW_CTR",MONITOR
18652,content_0173fb0dc986,4,"STALE_CONTENT,LOW_CTR",MONITOR


# 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [17]:
baseline[[
    "content_id",
    "score",
    "reason_code",
    "action"
]].head(20)


,content_id,score,reason_code,action
21268,content_0a91db491d14,6,"STALE_CONTENT,HIGH_IMPRESSIONS,LOW_CTR",REFRESH_CONTENT
16751,content_cf56e2e2e282,6,"STALE_CONTENT,HIGH_IMPRESSIONS,LOW_CTR",REFRESH_CONTENT
16514,content_7368877ea310,6,"STALE_CONTENT,HIGH_IMPRESSIONS,LOW_CTR",REFRESH_CONTENT
12045,content_c2d929d83eaa,6,"STALE_CONTENT,HIGH_IMPRESSIONS,LOW_CTR",REFRESH_CONTENT
11489,content_5feee3994adb,6,"STALE_CONTENT,HIGH_IMPRESSIONS,LOW_CTR",REFRESH_CONTENT
7021,content_1bfaa38ff26c,6,"STALE_CONTENT,HIGH_IMPRESSIONS,LOW_CTR",REFRESH_CONTENT
20924,content_277eeb6d46cc,4,"STALE_CONTENT,LOW_CTR",MONITOR
21201,content_afd26a07382d,4,"STALE_CONTENT,LOW_CTR",MONITOR
22980,content_f328d0e4e22b,4,"STALE_CONTENT,LOW_CTR",MONITOR
18652,content_0173fb0dc986,4,"STALE_CONTENT,LOW_CTR",MONITOR


| # | Action | Reason Code | Confidence | What would make it wrong? |
|---|--------|-------------|------------|----------------------------|
| 1 | Refresh Content | STALE_CONTENT, HIGH_IMPRESSIONS, LOW_CTR | High | CTR may be low because of strong competition rather than outdated content. |
| 2 | Refresh Content | STALE_CONTENT, HIGH_IMPRESSIONS, LOW_CTR | High | High impressions may already satisfy the page objective. |
| 3 | Refresh Content | STALE_CONTENT, HIGH_IMPRESSIONS, LOW_CTR | High | Seasonal traffic could affect the metrics. |
| 4 | Refresh Content | STALE_CONTENT, HIGH_IMPRESSIONS, LOW_CTR | High | The page may already be performing well for its target audience. |
| 5 | Refresh Content | STALE_CONTENT, HIGH_IMPRESSIONS, LOW_CTR | High | Low CTR could be caused by search intent mismatch. |
| 6 | Refresh Content | STALE_CONTENT, HIGH_IMPRESSIONS, LOW_CTR | High | SERP features may reduce CTR despite good content. |
| 7 | Monitor | STALE_CONTENT, LOW_CTR | Medium | Refresh may not improve performance if competition is the main issue. |
| 8 | Monitor | STALE_CONTENT, LOW_CTR | Medium | Low CTR alone is not enough evidence for refresh. |
| 9 | Monitor | STALE_CONTENT, LOW_CTR | Medium | Content may still satisfy user intent. |
|10 | Monitor | STALE_CONTENT, LOW_CTR | Medium | Ranking changes may be temporary. |
|11 | Monitor | STALE_CONTENT, LOW_CTR | Medium | Seasonal variation could explain the decline. |
|12 | Monitor | STALE_CONTENT, LOW_CTR | Medium | The page may have low search demand. |
|13 | Monitor | STALE_CONTENT, LOW_CTR | Medium | CTR may improve without content updates. |
|14 | Monitor | STALE_CONTENT, LOW_CTR | Medium | External events may have affected performance. |
|15 | Monitor | STALE_CONTENT, LOW_CTR | Medium | The page may already rank for the correct audience. |
|16 | Monitor | STALE_CONTENT, LOW_CTR | Medium | More historical data may change the decision. |
|17 | Monitor | STALE_CONTENT, LOW_CTR | Medium | Search trends may recover naturally. |
|18 | Monitor | STALE_CONTENT, LOW_CTR | Medium | Additional engagement metrics should be checked. |
|19 | Monitor | STALE_CONTENT, LOW_CTR | Medium | A manual review is recommended before refreshing. |
|20 | Monitor | STALE_CONTENT, LOW_CTR | Medium | Business context may justify keeping the page unchanged. |

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

### Weak Picks

Some high-scoring pages may not actually require a refresh. High impressions and low CTR can be influenced by strong competition, search intent, or seasonal trends rather than outdated content. These pages should be reviewed manually before taking action.

### Leakage Check

This baseline does not use future information or label-derived features. I intentionally excluded `trend_direction` and `trend_pct` because they are derived from the target. The rule only uses historical features such as `days_since_last_update`, `impressions_90d`, and `ctr`.

## Self-check

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.